In [ ]:
import boto3

s3 = boto3.resource(
    's3',
    endpoint_url='http://minio:9000',   # Use the container name "minio" (from docker-compose)
    aws_access_key_id='minio',
    aws_secret_access_key='minio123'
)

# Test bucket
bucket = s3.Bucket('test-bucket-4')
bucket.create()


In [ ]:
bucket.delete_bucket()

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("UnityCatalogTest")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    # Point spark_catalog itself at UC — no credential vending triggered
    .config("spark.sql.catalog.spark_catalog", "io.unitycatalog.spark.UCSingleCatalog")
    .config("spark.sql.catalog.spark_catalog.uri", "http://uc-server:8080")
    .config("spark.sql.catalog.spark_catalog.token", "test-token")
    # Spark handles MinIO directly
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

# Write directly via s3a (no UC credential vending involved)
data = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"])
data.write.format("delta").mode("overwrite").save("s3a://test-bucket/test/default/people")


In [ ]:

# Register in UC via REST API (as we did before)
import requests
requests.post(
    "http://uc-server:8080/api/2.1/unity-catalog/tables",
    headers={"Authorization": "Bearer test-token", "Content-Type": "application/json"},
    json={
        "name": "people",
        "catalog_name": "unity",
        "schema_name": "default",
        "table_type": "EXTERNAL",
        "data_source_format": "DELTA",
        "storage_location": "s3a://test-bucket/unity/default/people",  # use s3a here too
        "columns": [
            {"name": "id", "type_name": "INT", "type_text": "int",
             "type_json": '{"name":"id","type":"integer","nullable":true,"metadata":{}}', "position": 0, "nullable": True},
            {"name": "name", "type_name": "STRING", "type_text": "string",
             "type_json": '{"name":"name","type":"string","nullable":true,"metadata":{}}', "position": 1, "nullable": True}
        ]
    }
)

# Read back — Spark uses its own s3a credentials, UC just provides metadata
spark.sql("SELECT * FROM unity.default.people").show()

In [ ]:
from pyspark.sql import SparkSession
import requests

spark = (
    SparkSession.builder
    .appName("UnityCatalogTest")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.catalog.unity", "io.unitycatalog.spark.UCSingleCatalog")
    .config("spark.sql.catalog.unity.uri", "http://uc-server:8080")
    .config("spark.sql.catalog.unity.token", "test-token")
    .config("spark.sql.catalog.unity.renewCredential.enabled", "false")  # disable vending
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3.access.key", "minio")
    .config("spark.hadoop.fs.s3.secret.key", "minio123")
    .config("spark.hadoop.fs.s3.path.style.access", "true")
    .getOrCreate()
)


# Step 1: Write Delta table to MinIO via s3a
data = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"])
data.write.format("delta").mode("overwrite").save("s3a://test-bucket/unity/default/people")
print("✅ Written to MinIO")

# Step 2: Register in UC via REST API using s3a:// URI
response = requests.post(
    "http://uc-server:8080/api/2.1/unity-catalog/tables",
    headers={"Authorization": "Bearer test-token", "Content-Type": "application/json"},
    json={
        "name": "people",
        "catalog_name": "unity",
        "schema_name": "default",
        "table_type": "EXTERNAL",
        "data_source_format": "DELTA",
        "storage_location": "s3://test-bucket/unity/default/people",
        "columns": [
            {"name": "id", "type_name": "INT", "type_text": "int",
             "type_json": '{"name":"id","type":"integer","nullable":true,"metadata":{}}',
             "position": 0, "nullable": True},
            {"name": "name", "type_name": "STRING", "type_text": "string",
             "type_json": '{"name":"name","type":"string","nullable":true,"metadata":{}}',
             "position": 1, "nullable": True}
        ]
    }
)
print(f"UC registration: {response.status_code} - {response.json()}")

# Step 3: Read back through UC catalog
spark.sql("SELECT * FROM unity.default.people").show()

In [3]:
from pyspark.sql import SparkSession
import requests

# ─── Spark Session ───────────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("UnityCatalogTest")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

UC_URL = "http://uc-server:8080/api/2.1/unity-catalog"
HEADERS = {"Authorization": "Bearer test-token", "Content-Type": "application/json"}

# ─── UC Helper Functions ──────────────────────────────────────────────────────

def uc_register_table(catalog, schema, name, s3a_location, columns):
    """Register a table in UC. Uses s3:// internally (UC requirement)."""
    s3_location = s3a_location.replace("s3a://", "s3://")
    # Delete if exists
    requests.delete(f"{UC_URL}/tables/{catalog}.{schema}.{name}", headers=HEADERS)
    response = requests.post(f"{UC_URL}/tables", headers=HEADERS, json={
        "name": name,
        "catalog_name": catalog,
        "schema_name": schema,
        "table_type": "EXTERNAL",
        "data_source_format": "DELTA",
        "storage_location": s3_location,
        "columns": columns
    })
    assert response.status_code == 200, f"Registration failed: {response.json()}"
    print(f"✅ Registered {catalog}.{schema}.{name} in UC")
    return response.json()

def uc_read_table(catalog, schema, name):
    """Read a UC-registered table via Spark, bypassing credential vending."""
    response = requests.get(
        f"{UC_URL}/tables/{catalog}.{schema}.{name}", headers=HEADERS
    )
    assert response.status_code == 200, f"Table not found: {response.json()}"
    location = response.json()["storage_location"].replace("s3://", "s3a://")
    return spark.read.format("delta").load(location)

def uc_list_tables(catalog, schema):
    """List all tables in a UC schema."""
    response = requests.get(
        f"{UC_URL}/tables?catalog_name={catalog}&schema_name={schema}", headers=HEADERS
    )
    return [t["name"] for t in response.json().get("tables", [])]

def uc_create_catalog(catalog, comment=""):
    response = requests.post(f"{UC_URL}/catalogs", headers=HEADERS, json={
        "name": catalog,
        "comment": comment
    })
    assert response.status_code == 200, f"Catalog creation failed: {response.json()}"
    print(f"✅ Created catalog: {catalog}")
    return response.json()

def uc_create_schema(catalog, schema, comment=""):
    response = requests.post(f"{UC_URL}/schemas", headers=HEADERS, json={
        "name": schema,
        "catalog_name": catalog,
        "comment": comment
    })
    assert response.status_code == 200, f"Schema creation failed: {response.json()}"
    print(f"✅ Created schema: {catalog}.{schema}")
    return response.json()


def uc_delete_table(catalog, schema, name):
    response = requests.delete(
        f"{UC_URL}/tables/{catalog}.{schema}.{name}", headers=HEADERS
    )
    print(f"✅ Deleted {catalog}.{schema}.{name} from UC")


# ─── End-to-End Test ─────────────────────────────────────────────────────────

# 1. Write Delta table to MinIO
location = "s3a://test-bucket/test/default/people"
data = spark.createDataFrame([(1, "Alice"), (2, "Bob"), (3, "Charlie")], ["id", "name"])
data.write.format("delta").mode("overwrite").save(location)
print("✅ Written to MinIO")

uc_create_catalog("test")

uc_create_schema("test", "default")

# 2. Register in UC
uc_register_table(
    catalog="test", schema="default", name="people",
    s3a_location=location,
    columns=[
        {"name": "id", "type_name": "INT", "type_text": "int",
         "type_json": '{"name":"id","type":"integer","nullable":true,"metadata":{}}',
         "position": 0, "nullable": True},
        {"name": "name", "type_name": "STRING", "type_text": "string",
         "type_json": '{"name":"name","type":"string","nullable":true,"metadata":{}}',
         "position": 1, "nullable": True}
    ]
)

# 3. List tables via UC
print(f"Tables in unity.default: {uc_list_tables('test', 'default')}")

# 4. Read back through UC metadata + Spark storage
df = uc_read_table("test", "default", "people")
df.show()

✅ Written to MinIO
✅ Created catalog: test
✅ Created schema: test.default
✅ Registered test.default.people in UC
Tables in unity.default: ['people']
+---+-------+
| id|   name|
+---+-------+
|  3|Charlie|
|  1|  Alice|
|  2|    Bob|
+---+-------+

